In [2]:
# Remove unwanted warning
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
#warnings.simplefilter(action='ignore', catgeory=RuntimeWarning)

import os

# Data Management
import pandas as pd
import numpy as np
import polars as pl
import pyarrow as pa
from pandas_datareader.data import DataReader
from ta import add_all_ta_features

# Statistics
from statsmodels.tsa.stattools import adfuller

# Unsupervised Machine Learning
from sklearn.decomposition import PCA

# Supervised Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score

# Reporting
import matplotlib.pyplot as plt

In [3]:
# Global Variables
CSV_FILENAME = "stocks.csv"
PARQET_FILENAME = "stocks.parquet"
WORKING_DIR = "data/"
FEATURES = ["gvkey"]

### Data Extraction

In [19]:
if not os.path.exists(os.path.join(
        WORKING_DIR, CSV_FILENAME
    )):
    # read sample data
    file_path = os.path.join(
        WORKING_DIR, "ret_sample.csv"
    )
    raw = pl.read_csv(file_path)
    raw = raw.filter(pl.col("excntry").is_in(["CAN","USA"]))
    raw.write_csv(os.path.join(WORKING_DIR, CSV_FILENAME))

if not os.path.exists(PARQET_FILENAME):
    raw = pd.read_csv(os.path.join(WORKING_DIR, CSV_FILENAME), dtype={4: str})
    raw.to_parquet(PARQET_FILENAME, index=False, compression="snappy")


raw = pd.read_parquet(PARQET_FILENAME)




In [20]:

# Display basic information
print("DataFrame Head:")
print(raw.head())

DataFrame Head:
                id      date   ret_eom   gvkey  iid excntry  stock_ret  year  \
0  comp_001081_01C  20050228  20050228  1081.0  01C     CAN  -0.143457  2005   
1  comp_001096_01C  20050228  20050228  1096.0  01C     CAN   0.028077  2005   
2   comp_001117_02  20050228  20050228  1117.0   02     USA  -0.168627  2005   
3  comp_001186_01C  20050228  20050228  1186.0  01C     CAN   0.149056  2005   
4  comp_001243_01C  20050228  20050228  1243.0  01C     CAN   0.006239  2005   

   month  char_date  ...  betadown_252d  prc_highprc_252d  corr_1260d  \
0      2   20050131  ...       0.779315          0.672204    0.387781   
1      2   20050131  ...       0.445162          0.937664    0.245148   
2      2   20050131  ...       1.073565          0.708333    0.124188   
3      2   20050131  ...       1.326215          0.774557    0.174888   
4      2   20050131  ...       1.259809          0.764020    0.474511   

   betabab_1260d  rmax5_rvol_21d  age       qmj  qmj_prof  qmj_g

In [22]:

print("\nDataFrame Info:")
raw.info()



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1398807 entries, 0 to 1398806
Columns: 159 entries, id to qmj_safety
dtypes: float64(149), int64(7), object(3)
memory usage: 1.7+ GB


In [23]:

print("\nDataFrame Description:")
print(raw.describe(include='all'))


DataFrame Description:
                     id          date       ret_eom         gvkey      iid  \
count           1398807  1.398807e+06  1.398807e+06  1.398356e+06  1398356   
unique            16739           NaN           NaN           NaN       21   
top     comp_106995_01C           NaN           NaN           NaN       01   
freq                245           NaN           NaN           NaN  1021652   
mean                NaN  2.014337e+07  2.014337e+07  7.689555e+04      NaN   
std                 NaN  5.967078e+04  5.967080e+04  6.927447e+04      NaN   
min                 NaN  2.005020e+07  2.005023e+07  1.004000e+03      NaN   
25%                 NaN  2.009063e+07  2.009063e+07  1.826400e+04      NaN   
50%                 NaN  2.014073e+07  2.014073e+07  3.932500e+04      NaN   
75%                 NaN  2.020013e+07  2.020013e+07  1.434210e+05      NaN   
max                 NaN  2.025063e+07  2.025063e+07  3.562890e+05      NaN   

        excntry     stock_ret          

In [18]:
raw.columns

Index(['id', 'date', 'ret_eom', 'gvkey', 'iid', 'excntry', 'stock_ret', 'year',
       'month', 'char_date',
       ...
       'betadown_252d', 'prc_highprc_252d', 'corr_1260d', 'betabab_1260d',
       'rmax5_rvol_21d', 'age', 'qmj', 'qmj_prof', 'qmj_growth', 'qmj_safety'],
      dtype='object', length=159)